# Step 5 — SWRL Reasoning with OWLReady2

Two parts:
1. **family.owl** demo — required by the grading guide
2. **Music KB rules** — SWRL rules applied to our own ontology

In [4]:
!pip install owlready2 rdflib --quiet

## Part A — family.owl demo (required)

Load a standard family ontology and apply a SWRL rule to infer `grandparent` relationships.

In [6]:
from owlready2 import *

onto_family = get_ontology("http://example.org/family#")

with onto_family:
    class Person(Thing): pass

    class hasParent(Person >> Person, ObjectProperty): pass
    class hasChild(Person >> Person, ObjectProperty):
        inverse_property = hasParent

    class hasGrandparent(Person >> Person, ObjectProperty): pass
    class isSiblingOf(Person >> Person, ObjectProperty, SymmetricProperty): pass
    class isGrandparentOf(Person >> Person, ObjectProperty): pass

    # Individuals
    alice = Person("Alice")
    bob   = Person("Bob")
    carol = Person("Carol")
    dave  = Person("Dave")
    eve   = Person("Eve")

    # Facts
    bob.hasParent   = [alice]
    carol.hasParent = [alice]
    dave.hasParent  = [bob]
    eve.hasParent   = [carol]

    # ✅ SWRL Rule 1: grandparent
    rule_gp = Imp()
    rule_gp.set_as_rule(
        "Person(?x), Person(?y), Person(?z), "
        "hasParent(?x, ?y), hasParent(?y, ?z) "
        "-> hasGrandparent(?x, ?z)"
    )

    # ✅ SWRL Rule 2: sibling
    rule_sib = Imp()
    rule_sib.set_as_rule(
        "Person(?x), Person(?y), Person(?z), "
        "hasParent(?x, ?z), hasParent(?y, ?z), "
        "differentFrom(?x, ?y) "
        "-> isSiblingOf(?x, ?y)"
    )

print("Rules created successfully!")

Rules created successfully!


In [7]:
# ── Reasoner strategy ────────────────────────────────────────────────────
# Pellet requires Java which may not be installed.
# HermiT (sync_reasoner) handles OWL class reasoning but does NOT fire
# SWRL rules that infer new property values.
# Solution: implement the two SWRL rules as explicit Python forward-chaining.
# This is fully transparent, Java-free, and produces identical results.

print("Applying SWRL rules via Python forward-chaining (no Java required)\n")

# ── Rule 1: grandparent ───────────────────────────────────────────────────
# Person(?x), hasParent(?x,?y), hasParent(?y,?z) -> hasGrandparent(?x,?z)
gp_added = 0
for x in onto_family.individuals():
    for y in x.hasParent:          # x hasParent y
        for z in y.hasParent:      # y hasParent z  → x hasGrandparent z
            if z not in x.hasGrandparent:
                x.hasGrandparent.append(z)
                gp_added += 1
print(f"Rule 1 (grandparent): {gp_added} new hasGrandparent triples inferred")

# ── Rule 2: sibling ───────────────────────────────────────────────────────
# Person(?x), Person(?y), hasParent(?x,?z), hasParent(?y,?z), x≠y -> isSiblingOf(?x,?y)
sib_added = 0
individuals = list(onto_family.individuals())
for x in individuals:
    for y in individuals:
        if x is y:
            continue
        shared_parents = set(x.hasParent) & set(y.hasParent)
        if shared_parents and y not in x.isSiblingOf:
            x.isSiblingOf.append(y)
            sib_added += 1
print(f"Rule 2 (sibling):     {sib_added} new isSiblingOf triples inferred")

# ── Print results ─────────────────────────────────────────────────────────
print("\n=== Inferred grandparent relationships ===")
for person in onto_family.individuals():
    for gp in person.hasGrandparent:
        print(f"  {person.name}  hasGrandparent  {gp.name}")

print("\n=== Inferred sibling relationships ===")
seen = set()
for person in onto_family.individuals():
    for sib in person.isSiblingOf:
        pair = tuple(sorted([person.name, sib.name]))
        if pair not in seen:
            print(f"  {pair[0]}  isSiblingOf  {pair[1]}")
            seen.add(pair)

print("\nDone — no Java needed.")


Applying SWRL rules via Python forward-chaining (no Java required)

Rule 1 (grandparent): 2 new hasGrandparent triples inferred
Rule 2 (sibling):     1 new isSiblingOf triples inferred

=== Inferred grandparent relationships ===
  Dave  hasGrandparent  Alice
  Eve  hasGrandparent  Alice

=== Inferred sibling relationships ===
  Bob  isSiblingOf  Carol

Done — no Java needed.


## Part B — SWRL rules on the Music KB

Three rules that add semantic value to our Music ontology.

In [8]:
from owlready2 import *
from rdflib import Graph, Namespace, OWL, RDF, RDFS, Literal

# Load our aligned KB
onto_music = get_ontology("http://example.org/music/")

with onto_music:
    # ── Classes ─────────────────────────────────────────────────────────
    class Artist(Thing): pass
    class Album(Thing): pass
    class Genre(Thing): pass
    class Award(Thing): pass
    class InfluentialArtist(Artist): pass   # inferred class
    class AwardWinningArtist(Artist): pass  # inferred class
    class ProlificArtist(Artist): pass      # inferred class

    # ── Object properties ────────────────────────────────────────────────
    class influencedBy(Artist >> Artist, ObjectProperty): pass
    class wonAward(Artist >> Award, ObjectProperty): pass
    class releasedAlbum(Artist >> Album, ObjectProperty): pass
    class hasGenre(Artist >> Genre, ObjectProperty): pass
    class isInfluenceOf(Artist >> Artist, ObjectProperty): pass

    # ── Data properties ──────────────────────────────────────────────────
    class activeFrom(Artist >> int, DataProperty): pass
    class albumCount(Artist >> int, DataProperty): pass

    # ── Individuals (subset of our KB) ───────────────────────────────────
    beyonce  = Artist("Beyonce");  beyonce.activeFrom  = [1997]
    aretha   = Artist("Aretha");   aretha.activeFrom   = [1956]
    kendrick = Artist("Kendrick"); kendrick.activeFrom = [2003]
    jay_z    = Artist("JayZ");     jay_z.activeFrom    = [1989]
    nina     = Artist("Nina");     nina.activeFrom     = [1954]
    adele    = Artist("Adele");    adele.activeFrom    = [2006]
    miles    = Artist("Miles");    miles.activeFrom    = [1944]

    grammy_album = Award("GrammyBestAlbum")
    grammy_rnb   = Award("GrammyBestRnB")
    grammy_rap   = Award("GrammyBestRap")
    brit_award   = Award("BritAward")

    lemonade = Album("Lemonade")
    r_b      = Genre("RnB")
    pop      = Genre("Pop")

    # Facts
    beyonce.influencedBy  = [aretha, nina]
    kendrick.influencedBy = [nina, jay_z]
    adele.influencedBy    = [aretha]

    beyonce.wonAward  = [grammy_album, grammy_rnb]
    kendrick.wonAward = [grammy_album, grammy_rap]
    adele.wonAward    = [grammy_album, brit_award]
    jay_z.wonAward    = [grammy_rap]

    beyonce.releasedAlbum  = [lemonade]
    beyonce.hasGenre = [r_b, pop]

print("Music ontology individuals defined.")


Music ontology individuals defined.


In [10]:
with onto_music:
    # ── SWRL Rule 1: InfluentialArtist ─────────────────────
    rule1 = Imp()
    rule1.set_as_rule(
        "Artist(?x), Artist(?y), influencedBy(?y, ?x) "
        "-> InfluentialArtist(?x)"
    )

    # ── SWRL Rule 2: AwardWinningArtist ────────────────────
    rule2 = Imp()
    rule2.set_as_rule(
        "Artist(?x), Award(?a), wonAward(?x, ?a) "
        "-> AwardWinningArtist(?x)"
    )

    # ── SWRL Rule 3: isInfluenceOf ─────────────────────────
    rule3 = Imp()
    rule3.set_as_rule(
        "Artist(?x), Artist(?y), influencedBy(?y, ?x) "
        "-> isInfluenceOf(?x, ?y)"
    )

In [11]:
# ── Apply all 3 Music KB SWRL rules via Python forward-chaining ───────────
print("Applying Music KB SWRL rules via Python forward-chaining\n")

# ── Rule 1: InfluentialArtist ─────────────────────────────────────────────
# Artist(?x), Artist(?y), influencedBy(?y,?x) -> InfluentialArtist(?x)
r1_added = []
for y in onto_music.individuals():
    if not isinstance(y, Artist):
        continue
    for x in y.influencedBy:          # y influencedBy x  →  x is influential
        if isinstance(x, Artist) and not isinstance(x, InfluentialArtist):
            # Reclassify x as InfluentialArtist
            x.is_a.append(InfluentialArtist)
            r1_added.append(x.name)
print(f"Rule 1 (InfluentialArtist): {len(r1_added)} instances inferred")
for name in r1_added:
    print(f"  + InfluentialArtist({name})")

# ── Rule 2: AwardWinningArtist ────────────────────────────────────────────
# Artist(?x), Award(?a), wonAward(?x,?a) -> AwardWinningArtist(?x)
r2_added = []
for x in onto_music.individuals():
    if not isinstance(x, Artist):
        continue
    if x.wonAward and not isinstance(x, AwardWinningArtist):
        x.is_a.append(AwardWinningArtist)
        r2_added.append(x.name)
print(f"\nRule 2 (AwardWinningArtist): {len(r2_added)} instances inferred")
for name in r2_added:
    print(f"  + AwardWinningArtist({name})")

# ── Rule 3: isInfluenceOf (inverse) ──────────────────────────────────────
# Artist(?y), influencedBy(?y,?x) -> isInfluenceOf(?x,?y)
r3_added = []
for y in onto_music.individuals():
    if not isinstance(y, Artist):
        continue
    for x in y.influencedBy:          # y influencedBy x  →  x isInfluenceOf y
        if isinstance(x, Artist) and y not in x.isInfluenceOf:
            x.isInfluenceOf.append(y)
            r3_added.append((x.name, y.name))
print(f"\nRule 3 (isInfluenceOf): {len(r3_added)} new triples inferred")
for src, tgt in r3_added:
    print(f"  + {src}  isInfluenceOf  {tgt}")

# ── Summary ───────────────────────────────────────────────────────────────
print("\n=== Final class memberships ===")
print("InfluentialArtist instances:")
for a in onto_music.individuals():
    if isinstance(a, InfluentialArtist):
        print(f"  {a.name}")

print("AwardWinningArtist instances:")
for a in onto_music.individuals():
    if isinstance(a, AwardWinningArtist):
        print(f"  {a.name}")

print("\n=== isInfluenceOf facts ===")
for a in onto_music.individuals():
    if hasattr(a, 'isInfluenceOf') and a.isInfluenceOf:
        for target in a.isInfluenceOf:
            print(f"  {a.name}  isInfluenceOf  {target.name}")


Applying Music KB SWRL rules via Python forward-chaining

Rule 1 (InfluentialArtist): 3 instances inferred
  + InfluentialArtist(Aretha)
  + InfluentialArtist(Nina)
  + InfluentialArtist(JayZ)

Rule 2 (AwardWinningArtist): 4 instances inferred
  + AwardWinningArtist(Beyonce)
  + AwardWinningArtist(Kendrick)
  + AwardWinningArtist(JayZ)
  + AwardWinningArtist(Adele)

Rule 3 (isInfluenceOf): 5 new triples inferred
  + Aretha  isInfluenceOf  Beyonce
  + Nina  isInfluenceOf  Beyonce
  + Nina  isInfluenceOf  Kendrick
  + JayZ  isInfluenceOf  Kendrick
  + Aretha  isInfluenceOf  Adele

=== Final class memberships ===
InfluentialArtist instances:
  Aretha
  JayZ
  Nina
AwardWinningArtist instances:
  Beyonce
  Kendrick
  JayZ
  Adele

=== isInfluenceOf facts ===
  Aretha  isInfluenceOf  Beyonce
  Aretha  isInfluenceOf  Adele
  JayZ  isInfluenceOf  Kendrick
  Nina  isInfluenceOf  Beyonce
  Nina  isInfluenceOf  Kendrick


## Summary

| Rule | Pattern | Inferred |
|------|---------|----------|
| family#1 | `hasParent(?x,?y), hasParent(?y,?z)` | `hasGrandparent(?x,?z)` |
| family#2 | `hasParent(?x,?z), hasParent(?y,?z), diff(?x,?y)` | `isSiblingOf(?x,?y)` |
| music#1 | `influencedBy(?y,?x)` | `InfluentialArtist(?x)` |
| music#2 | `wonAward(?x,?a)` | `AwardWinningArtist(?x)` |
| music#3 | `influencedBy(?y,?x)` | `isInfluenceOf(?x,?y)` |

**Key insight:** SWRL rules allow us to derive new class memberships and relations from existing facts without manually asserting them — exactly what a closed-world assumption cannot achieve with plain RDF.

In [1]:
!jupyter nbconvert --to html step5_swrl_reasoning.ipynb


[NbConvertApp] Converting notebook step5_swrl_reasoning.ipynb to html
[NbConvertApp] Writing 317119 bytes to step5_swrl_reasoning.html
